# 02.5_build_dataset

This notebook builds a subject-level sleep feature dataset from multiple hypnogram files.

Pipeline:
- load many hypnogram CSV files
- extract subject-level sleep features
- combine all rows into one dataset
- save `sleep_features_all_subjects.csv`

Output:
- participant-level feature table ready for downstream MESA merge

In [ ]:
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    raise FileNotFoundError('Project root not found. Make sure README.md and data/ exist.')

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR =', RAW_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

csv_files = [PROCESSED_DIR / '../data/processed/hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

In [ ]:
INPUT_DIR = Path('hypnograms')
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [ ]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:10]:
    print(f.name)

In [ ]:
from pathlib import Path

INPUT_DIR = Path('hypnograms')
INPUT_DIR.mkdir(exist_ok=True)

print("Created or already exists:", INPUT_DIR.resolve())

In [ ]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

In [ ]:
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders here:")
for p in Path('.').iterdir():
    print(p)

In [ ]:
INPUT_DIR = RAW_DIR / 'sleep_edf'
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [ ]:
csv_files = [PROCESSED_DIR / '../data/processed/hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

In [ ]:
test_file = csv_files[0]
print('Testing file:', test_file.name)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
test_features

In [ ]:
INPUT_DIR = Path('hypnograms')
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [ ]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:10]:
    print(f.name)

In [ ]:
def clean_hypnogram(df):
    df = df.copy()
    df = df[~df['description'].str.contains('?', regex=False, na=False)].copy()
    df = df[df['duration'] > 0].copy()
    df = df.reset_index(drop=True)
    return df

In [ ]:
def find_sleep_boundaries(df):
    sleep_mask = df['description'] != 'Sleep stage W'

    if sleep_mask.sum() == 0:
        return None, None

    first_sleep_idx = df[sleep_mask].index[0]
    last_sleep_idx = df[sleep_mask].index[-1]

    return first_sleep_idx, last_sleep_idx

In [ ]:
def extract_sleep_features(hypno_df, subject_id=None):
    df = clean_hypnogram(hypno_df)

    total_recording_time_sec = df['duration'].sum()

    first_sleep_idx, last_sleep_idx = find_sleep_boundaries(df)

    if first_sleep_idx is None:
        return pd.DataFrame([{
            'subject_id': subject_id,
            'recording_hours': total_recording_time_sec / 3600,
            'sleep_period_hours': np.nan,
            'total_sleep_hours': 0,
            'sleep_latency_min': np.nan,
            'rem_latency_min': np.nan,
            'fragmentation': np.nan,
            'wake_pct_in_sleep_period': np.nan,
            'rem_pct_of_tst': np.nan,
            'n1_pct_of_tst': np.nan,
            'n2_pct_of_tst': np.nan,
            'deep_sleep_pct_of_tst': np.nan,
            'sleep_efficiency': np.nan
        }])

    sleep_latency_sec = df.loc[:first_sleep_idx - 1, 'duration'].sum() if first_sleep_idx > 0 else 0

    rem_indices = df[df['description'] == 'Sleep stage R'].index
    rem_after_sleep = rem_indices[rem_indices >= first_sleep_idx]

    if len(rem_after_sleep) > 0:
        first_rem_idx = rem_after_sleep[0]
        rem_latency_sec = df.loc[first_sleep_idx:first_rem_idx - 1, 'duration'].sum() if first_rem_idx > first_sleep_idx else 0
    else:
        rem_latency_sec = np.nan

    sleep_period_df = df.loc[first_sleep_idx:last_sleep_idx].copy().reset_index(drop=True)
    sleep_period_time_sec = sleep_period_df['duration'].sum()

    wake_in_sleep_period_sec = sleep_period_df.loc[
        sleep_period_df['description'] == 'Sleep stage W', 'duration'
    ].sum()

    total_sleep_time_sec = sleep_period_time_sec - wake_in_sleep_period_sec

    sleep_only_df = sleep_period_df[sleep_period_df['description'] != 'Sleep stage W'].copy()
    sleep_stage_time = sleep_only_df.groupby('description')['duration'].sum()

    if total_sleep_time_sec > 0:
        sleep_stage_pct = sleep_stage_time / total_sleep_time_sec * 100
    else:
        sleep_stage_pct = pd.Series(dtype=float)

    fragmentation = (sleep_period_df['description'] != sleep_period_df['description'].shift()).sum()

    features = {
        'subject_id': subject_id,
        'recording_hours': total_recording_time_sec / 3600,
        'sleep_period_hours': sleep_period_time_sec / 3600,
        'total_sleep_hours': total_sleep_time_sec / 3600,
        'sleep_latency_min': sleep_latency_sec / 60,
        'rem_latency_min': rem_latency_sec / 60 if pd.notna(rem_latency_sec) else np.nan,
        'fragmentation': fragmentation,
        'wake_pct_in_sleep_period': (
            wake_in_sleep_period_sec / sleep_period_time_sec * 100
            if sleep_period_time_sec > 0 else np.nan
        ),
        'rem_pct_of_tst': sleep_stage_pct.get('Sleep stage R', 0),
        'n1_pct_of_tst': sleep_stage_pct.get('Sleep stage 1', 0),
        'n2_pct_of_tst': sleep_stage_pct.get('Sleep stage 2', 0),
        'deep_sleep_pct_of_tst': (
            sleep_stage_pct.get('Sleep stage 3', 0) +
            sleep_stage_pct.get('Sleep stage 4', 0)
        ),
        'sleep_efficiency': (
            total_sleep_time_sec / sleep_period_time_sec
            if sleep_period_time_sec > 0 else np.nan
        )
    }

    return pd.DataFrame([features])

In [ ]:
def load_hypnogram_csv(file_path):
    df = pd.read_csv(file_path)

    if 'duration' not in df.columns and 'duration_sec' in df.columns:
        df = df.rename(columns={'duration_sec': 'duration'})

    required_cols = {'description', 'duration'}
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    df = df[['description', 'duration']].copy()
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')

    return df

In [ ]:
def parse_subject_id(file_path):
    return file_path.stem

In [ ]:
test_file = csv_files[0]
print('Testing file:', test_file.name)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
test_features

In [ ]:
from pathlib import Path

INPUT_DIR = RAW_DIR / 'sleep_edf'
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [ ]:
csv_files = [PROCESSED_DIR / '../data/processed/hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

In [ ]:
if len(csv_files) == 0:
    print("No CSV files found. Re-run the path/file discovery cells.")
else:
    test_file = csv_files[0]
    print('Testing file:', test_file.name)

    test_hypno_df = load_hypnogram_csv(test_file)
    display(test_hypno_df.head())

    test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
    display(test_features)

In [ ]:
all_features = []
processing_log = []

for file_path in csv_files:
    subject_id = parse_subject_id(file_path)

    try:
        hypno_df = load_hypnogram_csv(file_path)
        features_df = extract_sleep_features(hypno_df, subject_id=subject_id)

        all_features.append(features_df)

        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'success',
            'n_rows': len(hypno_df),
            'message': ''
        })

    except Exception as e:
        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'failed',
            'n_rows': np.nan,
            'message': str(e)
        })

sleep_features_df = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
processing_log_df = pd.DataFrame(processing_log)

In [ ]:
print('sleep_features_df shape:', sleep_features_df.shape)
display(sleep_features_df.head())

print('processing_log_df shape:', processing_log_df.shape)
display(processing_log_df.head(20))

In [ ]:
processing_log_df['status'].value_counts(dropna=False)

In [ ]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

In [ ]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

In [ ]:
from pathlib import Path

INPUT_DIR = RAW_DIR / 'sleep_edf'
csv_files = [PROCESSED_DIR / '../data/processed/hypno_df.csv']

print("csv_files:", csv_files)
print("len(csv_files):", len(csv_files))

In [ ]:
test_file = csv_files[0]
print("Testing file:", test_file)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
display(test_features)

In [ ]:
all_features = []
processing_log = []

for file_path in csv_files:
    subject_id = parse_subject_id(file_path)

    try:
        hypno_df = load_hypnogram_csv(file_path)
        features_df = extract_sleep_features(hypno_df, subject_id=subject_id)

        all_features.append(features_df)

        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'success',
            'n_rows': len(hypno_df),
            'message': ''
        })

    except Exception as e:
        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'failed',
            'n_rows': np.nan,
            'message': str(e)
        })

sleep_features_df = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
processing_log_df = pd.DataFrame(processing_log)

print("sleep_features_df shape:", sleep_features_df.shape)
print("processing_log_df shape:", processing_log_df.shape)

display(sleep_features_df)
display(processing_log_df)

In [ ]:
print('sleep_features_df shape:', sleep_features_df.shape)
display(sleep_features_df.head())

print('processing_log_df shape:', processing_log_df.shape)
display(processing_log_df.head(20))

In [ ]:
processing_log_df['status'].value_counts(dropna=False)

In [ ]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

In [ ]:
sleep_features_df.to_csv(PROCESSED_DIR / 'sleep_features_all_subjects.csv', index=False)
processing_log_df.to_csv(PROCESSED_DIR / 'processing_log.csv', index=False)

In [ ]:
check_df = pd.read_csv(PROCESSED_DIR / 'sleep_features_all_subjects.csv')
check_df